# 06 · Tuning, model registry and the test set

LightGBM won the comparison in notebook 05 (PR-AUC 0.494 with 100 trees and default settings). Now we make it as good as possible, register it and test it once.

| Step | What happens | MLflow tag `stage` |
| --- | --- | --- |
| 1 · Early stopping | up to 5,000 trees with a smaller learning rate; training stops when `valid` stops improving | `tuning` |
| 2 · Hyperparameter tuning | Optuna tries 30 combinations of settings, each with early stopping | `tuning` (parent), `tuning-trial` (each try) |
| 3 · Final model and registry | train with the best settings; register it as `@champion` only if it beats the default LightGBM | `champion-candidate` |
| 4 · Test set | load `@champion` from the registry, evaluate **once** on July 2026, per train group and for calibration | – |

**Why `valid` is not enough for the final score:** early stopping, tuning and the registration decision all look at `valid`. After 30 tries, the best result on `valid` is partly luck, so it is slightly too optimistic. `test` has never influenced any decision; only its score is an honest estimate.

Expected run time: early stopping about 1 minute, tuning up to 45 minutes (stops after 30 tries or 45 minutes), the rest a few minutes. The MLflow server must be running.

In [1]:
import os
import time

import lightgbm as lgb
import mlflow
import mlflow.sklearn
import numpy as np
import optuna
import pandas as pd
import polars as pl
from lightgbm import LGBMClassifier
from mlflow import MlflowClient
from sklearn.pipeline import Pipeline

from db_delay_mlops.features.columns import FEATURES, TARGET, TEXT_FEATURES
from db_delay_mlops.models.evaluate import evaluate
from db_delay_mlops.models.preprocessing import CategoryCaster
from db_delay_mlops.paths import DATA_DIR

mlflow.set_tracking_uri(os.environ.get("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000"))
mlflow.set_experiment("db-delay-first-experiment")

MODEL_NAME = "db-delay-lightgbm"  # name in the model registry
SEED = 42
# Types skops may load: our own transformer and LightGBM's classes (see docs/baselines-und-vorverarbeitung.md)
TRUSTED_TYPES = [
    "collections.OrderedDict",
    "db_delay_mlops.models.preprocessing.CategoryCaster",
    "lightgbm.basic.Booster",
    "lightgbm.sklearn.LGBMClassifier",
]

## Data

Only `train` and `valid` are loaded here. `test` is loaded in the last section, so it cannot be used by accident.

**Early stopping needs `valid` already prepared.** A pipeline only prepares the data passed to `fit()`, not the validation data handed to the model inside it. So `valid` goes through a `CategoryCaster` that learned its categories on **`train`**: then "Köln Hbf" has the same internal number in both tables. Learning the categories on `valid` itself would give a different numbering and silently mix up stations.

In [2]:
data_dir = DATA_DIR / "processed" / "first_experiment"
train = pl.read_parquet(data_dir / "train.parquet")
valid = pl.read_parquet(data_dir / "valid.parquet")

X_train, y_train = train.select(FEATURES).to_pandas(), train[TARGET].to_numpy()
X_valid, y_valid = valid.select(FEATURES).to_pandas(), valid[TARGET].to_numpy()

# Categories learned on train only, applied to valid (for early stopping)
X_valid_prepared = CategoryCaster(columns=TEXT_FEATURES).fit(X_train).transform(X_valid)
print(X_train.shape, X_valid.shape)

(1120128, 13) (285416, 13)


## Training function

Your `train_and_log` from notebook 05, extended for early stopping:

- `model__eval_X=...`, `model__eval_y=...`: in a scikit-learn pipeline, `<step name>__<argument>` passes an argument to one step. Here: "give the step called `model` the validation data". (`eval_X`/`eval_y` replace the older `eval_set` since LightGBM 4.7.)
- `lgb.early_stopping(100)`: after every tree LightGBM checks the log loss on `valid`; if it has not improved for 100 trees, training stops and the best tree count is kept. We stop on log loss because it checks the probabilities themselves; Optuna then picks settings by PR-AUC.
- `best_iteration` is logged as a **metric**: the maximum number of trees is a setting, the number actually used is a result.
- `log_model=False` for the 30 tuning tries: only the final model is saved, otherwise `mlartifacts/` fills up with 30 models nobody needs.
- `nested=...`: inside the Optuna parent run, each try becomes a child run, so the 30 tries are grouped in the MLflow UI.

In [3]:
def make_lgbm_pipeline(**lgbm_params) -> Pipeline:
    """CategoryCaster + LightGBM, with fixed seed and no console output."""
    return Pipeline([
        ("categories", CategoryCaster(columns=TEXT_FEATURES)),  # text -> category (learned on train)
        ("model", LGBMClassifier(random_state=SEED, verbose=-1, **lgbm_params)),
    ])


def train_and_log(
    run_name: str,
    pipeline: Pipeline,
    stage: str,
    early_stopping_rounds: int = 100,
    log_model: bool = True,
) -> tuple[dict[str, float], mlflow.models.model.ModelInfo | None]:
    """Fit with early stopping on valid, evaluate on valid and log one MLflow run.

    Returns the metrics and, if the model was saved, its ModelInfo (needed to register it).
    """
    model = pipeline[-1]  # the last step of the pipeline is the model itself

    with mlflow.start_run(run_name=run_name, nested=mlflow.active_run() is not None):
        start = time.perf_counter()
        pipeline.fit(
            X_train,
            y_train,
            model__eval_X=X_valid_prepared,  # "model__" = passed to the step called "model"
            model__eval_y=y_valid,
            model__callbacks=[lgb.early_stopping(early_stopping_rounds, verbose=False)],
        )
        fit_seconds = time.perf_counter() - start

        p_valid = pipeline.predict_proba(X_valid)[:, 1]  # uses the best number of trees automatically
        metrics = evaluate(y_valid, p_valid) | {"fit_seconds": fit_seconds, "best_iteration": model.best_iteration_}

        mlflow.set_tag("stage", stage)
        mlflow.log_params({
            "model": type(model).__name__,
            "features": ",".join(FEATURES),
            "n_train": len(X_train),
            "early_stopping_rounds": early_stopping_rounds,
        })
        mlflow.log_params(model.get_params())
        mlflow.log_metrics(metrics)

        model_info = None
        if log_model:
            model_info = mlflow.sklearn.log_model(
                pipeline,
                name="model",
                input_example=X_valid.head(5),
                skops_trusted_types=TRUSTED_TYPES,
            )

    print(f"{run_name}: {fit_seconds:.0f} s, {model.best_iteration_} trees, pr_auc={metrics['pr_auc']:.4f}, brier={metrics['brier']:.4f}")
    return metrics, model_info

## 1 · Early stopping

Same model as the default LightGBM, but a smaller learning rate (0.05 instead of 0.1) and up to 5,000 trees. Smaller steps usually give a better model, but need more trees; early stopping finds out how many.

In [4]:
es_metrics, _ = train_and_log(
    run_name="lightgbm-early-stopping",
    pipeline=make_lgbm_pipeline(n_estimators=5000, learning_rate=0.05),
    stage="tuning",
)

/home/kleve/projects/db-delay-mlops/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
/home/kleve/projects/db-delay-mlops/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing value

🏃 View run lightgbm-early-stopping at: http://127.0.0.1:5000/#/experiments/1/runs/72b0b530081840bbb418750f087f271f
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
lightgbm-early-stopping: 17 s, 360 trees, pr_auc=0.4966, brier=0.1281


## 2 · Hyperparameter tuning with Optuna

**Is tuning worth it?** Usually it brings less than more data or better features, but it is cheap with LightGBM and shows a clean, logged search. We keep it small: 30 tries.

Optuna does not try combinations at random. After a few tries it learns which areas of the settings look promising and tries more there (TPE sampler).

| Setting | What it controls | Range |
| --- | --- | --- |
| `learning_rate` | how big each correction step is | 0.02–0.2 |
| `num_leaves` | how complex one tree can be | 15–255 |
| `min_child_samples` | minimum stops in a leaf; higher = fewer rules learned from small groups | 20–500 |
| `colsample_bytree` | share of features each tree may use | 0.5–1.0 |
| `subsample` | share of rows each tree sees | 0.5–1.0 |
| `reg_lambda` | penalty for extreme leaf values | 0.001–10 |

`log=True` means: try small values as often as large ones (0.02–0.04 gets as many tries as 0.1–0.2).

In [5]:
optuna.logging.set_verbosity(optuna.logging.WARNING)  # one line per try is enough, our function prints it


def objective(trial: optuna.Trial) -> float:
    """One try: settings suggested by Optuna -> train with early stopping -> PR-AUC on valid."""
    params = {
        "n_estimators": 5000,  # upper limit, early stopping decides
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.2, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 15, 255, log=True),
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 500, log=True),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "subsample_freq": 1,  # needed, otherwise LightGBM ignores subsample
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
    }
    metrics, _ = train_and_log(
        run_name=f"trial-{trial.number:02d}",
        pipeline=make_lgbm_pipeline(**params),
        stage="tuning-trial",
        log_model=False,
    )
    trial.set_user_attr("best_iteration", metrics["best_iteration"])
    return metrics["pr_auc"]

In [6]:
N_TRIALS = 30

with mlflow.start_run(run_name="optuna-lightgbm"):
    mlflow.set_tag("stage", "tuning")
    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=N_TRIALS, timeout=45 * 60)  # stop after 30 tries or 45 minutes

    mlflow.log_params({f"best_{k}": v for k, v in study.best_params.items()} | {"n_trials": len(study.trials)})
    mlflow.log_metric("best_pr_auc", study.best_value)

print(f"best PR-AUC on valid: {study.best_value:.4f} (try {study.best_trial.number})")

# The 10 best tries: PR-AUC, trees kept by early stopping, settings
trials = study.trials_dataframe()
param_columns = [c for c in trials.columns if c.startswith("params_")]
trials.sort_values("value", ascending=False)[["number", "value", "user_attrs_best_iteration", *param_columns]].head(10).round(4)

🏃 View run trial-00 at: http://127.0.0.1:5000/#/experiments/1/runs/ed2a5130fc154163a3f9ea4b1b9ca6f0
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
trial-00: 19 s, 127 trees, pr_auc=0.5029, brier=0.1273
🏃 View run trial-01 at: http://127.0.0.1:5000/#/experiments/1/runs/ef03edc8e4c9411e847161c277f16832
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
trial-01: 34 s, 411 trees, pr_auc=0.5050, brier=0.1271
🏃 View run trial-02 at: http://127.0.0.1:5000/#/experiments/1/runs/fa74cb50e7764e8388710134fcf7a775
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
trial-02: 28 s, 355 trees, pr_auc=0.4990, brier=0.1279
🏃 View run trial-03 at: http://127.0.0.1:5000/#/experiments/1/runs/0b5d453e2b574a278836e814017fb2c6
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
trial-03: 37 s, 785 trees, pr_auc=0.5005, brier=0.1276
🏃 View run trial-04 at: http://127.0.0.1:5000/#/experiments/1/runs/52affd464b7a40e6b9b08492e1c865bb
🧪 View experiment at: http://127.0.0.1:5

,number,value,user_attrs_best_iteration,params_colsample_bytree,params_learning_rate,params_min_child_samples,params_num_leaves,params_reg_lambda,params_subsample
28,28,0.5053,218,0.7742,0.0336,56,238,3.5918,0.5511
22,22,0.5052,408,0.9209,0.0216,68,184,2.1385,0.5064
7,7,0.5051,413,0.8313,0.0216,45,197,0.1203,0.6559
1,1,0.5050,411,0.8540,0.0229,137,174,7.5795,0.5103
27,27,0.5047,222,0.8631,0.0406,57,190,1.5964,0.6345
11,11,0.5047,409,0.8615,0.0251,79,156,1.1465,0.5077
29,29,0.5046,222,0.9372,0.0365,26,163,0.4853,0.5711
13,13,0.5044,472,0.7043,0.0203,43,195,0.4296,0.6193
16,16,0.5043,330,0.9315,0.0222,41,182,0.0125,0.5035
24,24,0.5036,230,0.8896,0.0204,101,220,9.5160,0.7406


## 3 · Final model and model registry

Train once more with the best settings, this time **saving** the model.

**Quality gate:** the model is only registered as `@champion` if it beats the default LightGBM from notebook 05 on `valid`. The bar is read from MLflow, not typed in by hand, so it is always the current value. This is the same rule the automatic retraining will use later: a new model replaces the champion only if it is better.

**Registry vocabulary:**

| Term | Meaning |
| --- | --- |
| Registered model | a name in the registry, here `db-delay-lightgbm` |
| Version | every registration creates a new version: 1, 2, 3, … |
| Alias | a movable pointer to one version, here `@champion`. The API loads `models:/db-delay-lightgbm@champion`; after retraining only the alias moves, the API code stays the same. |

In [7]:
best_params = study.best_params | {"n_estimators": 5000, "subsample_freq": 1}
final_metrics, model_info = train_and_log(
    run_name="lightgbm-tuned",
    pipeline=make_lgbm_pipeline(**best_params),
    stage="champion-candidate",
)

# The bar: best default LightGBM from notebook 05
bar = mlflow.search_runs(
    filter_string="tags.stage = 'candidate' AND tags.mlflow.runName = 'lightgbm' AND attributes.status = 'FINISHED'"
)["metrics.pr_auc"].max()
print(f"tuned: {final_metrics['pr_auc']:.4f} | default LightGBM: {bar:.4f} | early stopping only: {es_metrics['pr_auc']:.4f}")

/home/kleve/projects/db-delay-mlops/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
/home/kleve/projects/db-delay-mlops/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing value

🏃 View run lightgbm-tuned at: http://127.0.0.1:5000/#/experiments/1/runs/819df21387054c58998e48bc37016f83
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
lightgbm-tuned: 24 s, 218 trees, pr_auc=0.5053, brier=0.1270
tuned: 0.5053 | default LightGBM: 0.4942 | early stopping only: 0.4966


In [8]:
client = MlflowClient()

if final_metrics["pr_auc"] > bar:
    version = mlflow.register_model(model_info.model_uri, MODEL_NAME)  # creates the next version
    client.set_registered_model_alias(MODEL_NAME, "champion", version.version)  # move the pointer
    client.set_model_version_tag(MODEL_NAME, version.version, "valid_pr_auc", f"{final_metrics['pr_auc']:.4f}")
    print(f"registered {MODEL_NAME} version {version.version} as @champion")
else:
    print("not better than the default LightGBM: nothing registered, the champion stays")

Successfully registered model 'db-delay-lightgbm'.
2026/11/08 11:26:11 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: db-delay-lightgbm, version 1
Created version '1' of model 'db-delay-lightgbm'.


registered db-delay-lightgbm version 1 as @champion


## 4 · Test set: once, at the very end

We load the model **from the registry by its alias**, exactly as the API will do later, and not the Python object from above. If this works, the registered model is complete: pipeline, categories, signature.

Three views on the test set:
1. **Overall** compared with `valid`: how much worse is it two years later (drift)?
2. **Per train group**: an overall score mostly describes the S-Bahn (almost half of all arrivals).
3. **Calibration**: group the predictions into bins (0–10 %, 10–20 %, …) and compare the mean prediction with the actual late share. For a well-calibrated model both columns are close.

The results are logged into the run of the final model as `test_*` metrics and as tables.

In [9]:
test = pl.read_parquet(data_dir / "test.parquet")
X_test, y_test = test.select(FEATURES).to_pandas(), test[TARGET].to_numpy()

champion = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}@champion")
p_test = champion.predict_proba(X_test)[:, 1]
test_metrics = evaluate(y_test, p_test)

pd.DataFrame({"valid (July 2024)": final_metrics, "test (July 2026)": test_metrics}).drop(["fit_seconds", "best_iteration"]).round(4)

,valid (July 2024),test (July 2026)
pr_auc,0.5053,0.5041
roc_auc,0.7972,0.7774
brier,0.1270,0.1493
log_loss,0.4006,0.4587


In [10]:
# 2. Per train group
groups = test["train_group"].to_numpy()
by_group = pd.DataFrame([
    {"train_group": g, "arrivals": int((groups == g).sum()), "late_share": y_test[groups == g].mean(), **evaluate(y_test[groups == g], p_test[groups == g])}
    for g in ["long_distance", "regional", "s_bahn"]
]).round(4)

# 3. Calibration: predicted vs. actual late share in 10 bins
calibration = (
    pl.DataFrame({"p": p_test, "y": y_test})
    .with_columns(bin=(pl.col("p") * 10).floor().clip(0, 9) / 10)  # 0.0, 0.1, ..., 0.9
    .group_by("bin")
    .agg(arrivals=pl.len(), mean_predicted=pl.col("p").mean(), actual_late_share=pl.col("y").mean())
    .sort("bin")
    .to_pandas()
    .round(3)
)

with mlflow.start_run(run_id=model_info.run_id):  # add to the run of the final model
    mlflow.log_metrics({f"test_{k}": v for k, v in test_metrics.items()})
    mlflow.log_table(by_group, artifact_file="test_by_train_group.json")
    mlflow.log_table(calibration, artifact_file="test_calibration.json")

display(by_group)
calibration

🏃 View run lightgbm-tuned at: http://127.0.0.1:5000/#/experiments/1/runs/819df21387054c58998e48bc37016f83
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


,train_group,arrivals,late_share,pr_auc,roc_auc,brier,log_loss
0,long_distance,152664,0.5365,0.6829,0.6721,0.2373,0.6706
1,regional,538635,0.2792,0.4454,0.6901,0.1868,0.5573
2,s_bahn,635649,0.1240,0.3269,0.7684,0.0965,0.3241


,bin,arrivals,mean_predicted,actual_late_share
0,0.0,516914,0.051,0.066
1,0.1,306524,0.145,0.201
2,0.2,183116,0.247,0.310
3,0.3,118786,0.347,0.396
4,0.4,82527,0.447,0.482
5,0.5,57631,0.547,0.549
6,0.6,36645,0.646,0.611
7,0.7,18526,0.743,0.707
8,0.8,5693,0.839,0.772
9,0.9,586,0.918,0.792


## Questions

1. How many trees did early stopping keep? What would happen with 5,000 trees and no early stopping?
2. How much did tuning add compared with early stopping alone? Was it worth 30 tries?
3. The test PR-AUC is probably lower than on `valid`. Name two reasons. (Tip: compare the late shares of July 2024 and July 2026, and think about how often we looked at `valid`.)
4. Which train group does the model predict best, and which worst? Does that match what you saw on the map?
5. Is the model well calibrated? Where are predicted and actual late share furthest apart?